In [2]:
! pip install -q langchain==0.1.20 langchain-community==0.0.38 langchain-text-splitters sentence-transformers faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
twine 6.1.0 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import os

# Fixed imports for newer LangChain versions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ All imports successful")

✅ All imports successful


In [2]:
with open("data/raw_pages.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

print(f"✅ Loaded {len(pages)} pages")
for p in pages:
    print(f"  - {p['title']} | {len(p['content'])} chars")

✅ Loaded 6 pages
  - CrocoIT - CrocoIT | 2861 chars
  - About Us – CrocoIT | 2597 chars
  - Solutions – CrocoIT | 1974 chars
  - Our Products – CrocoIT | 1681 chars
  - Articles – CrocoIT | 1500 chars
  - Case Studies – CrocoIT | 1593 chars


In [3]:
def clean_content(text):
    lines = text.split("\n")
    seen_lines = set()
    cleaned = []

    for line in lines:
        line = line.strip()
        if len(line) < 20:
            continue
        if line in seen_lines:
            continue
        seen_lines.add(line)
        cleaned.append(line)

    return "\n".join(cleaned)

# Apply cleaning to all pages
for page in pages:
    original_len = len(page["content"])
    page["content"] = clean_content(page["content"])
    print(f"✅ {page['title']}: {original_len} → {len(page['content'])} chars")

✅ CrocoIT - CrocoIT: 2861 → 2608 chars
✅ About Us – CrocoIT: 2597 → 2498 chars
✅ Solutions – CrocoIT: 1974 → 1974 chars
✅ Our Products – CrocoIT: 1681 → 1615 chars
✅ Articles – CrocoIT: 1500 → 1032 chars
✅ Case Studies – CrocoIT: 1593 → 1203 chars


In [4]:
documents = []

for page in pages:
    doc = Document(
        page_content=page["content"],
        metadata={
            "url": page["url"],
            "title": page["title"]
        }
    )
    documents.append(doc)

print(f"✅ Created {len(documents)} documents")

✅ Created 6 documents


In [5]:
def clean_chunk(text):
    lines = text.split("\n")
    seen = set()
    result = []

    for line in lines:
        line = line.strip()
        if len(line) < 20:
            continue
        if line in seen:
            continue
        seen.add(line)
        result.append(line)

    return "\n".join(result)


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

raw_chunks = splitter.split_documents(documents)

# Clean each chunk
for chunk in raw_chunks:
    chunk.page_content = clean_chunk(chunk.page_content)

# Remove short chunks
chunks = [c for c in raw_chunks if len(c.page_content.strip()) > 100]

print(f"✅ Total chunks after cleaning  : {len(chunks)}")
print(f"\n--- Preview of first chunk ---")
print(chunks[0].page_content)
print(f"\nSource: {chunks[0].metadata['url']}")

✅ Total chunks after cleaning  : 28

--- Preview of first chunk ---
Unlock Insights and Secure Loyalty
Every business deserves a fighting chance of success, and that’s regardless of the type of business it is.
We create visually stunning, user-focused designs that elevate your brand and deliver seamless experiences across all devices.
We craft beautiful, readable typography systems that enhance your brand identity and ensure clarity across every platform.

Source: https://crocoit.com


In [6]:
seen = set()
unique_chunks = []

for chunk in chunks:
    # Use stripped content as the unique key
    content_key = chunk.page_content.strip()
    
    if content_key not in seen:
        seen.add(content_key)
        unique_chunks.append(chunk)

print(f"Original chunks  : {len(chunks)}")
print(f"After dedup      : {len(unique_chunks)}")
print(f"Duplicates removed: {len(chunks) - len(unique_chunks)}")

# Replace chunks with deduplicated version
chunks = unique_chunks

Original chunks  : 28
After dedup      : 26
Duplicates removed: 2


In [7]:
def is_near_duplicate(text1, text2, threshold=0.9):
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    if not words1 or not words2:
        return False
    overlap = len(words1 & words2) / min(len(words1), len(words2))
    return overlap >= threshold

final_chunks = []
for chunk in chunks:
    is_dup = False
    for existing in final_chunks:
        if is_near_duplicate(chunk.page_content, existing.page_content):
            is_dup = True
            break
    if not is_dup:
        final_chunks.append(chunk)

print(f"After near-dedup : {len(final_chunks)}")
chunks = final_chunks

After near-dedup : 25


In [8]:
# Cell 8.5 — Remove footer/copyright/navigation noise
noise_keywords = [
    "all rights reserved",
    "privacy policy",
    "terms of service",
    "developed by",
    "copyright",
    "© 20",
    "cookie",
    "subscribe to our newsletter",
    "follow us",
    "all rights",
]

def is_noise(text):
    text_lower = text.lower()
    # If more than 2 noise keywords found → it's noise
    count = sum(1 for kw in noise_keywords if kw in text_lower)
    return count >= 2

clean_final = []
removed = 0

for chunk in chunks:
    if is_noise(chunk.page_content):
        removed += 1
        print(f"🗑️ Removed noise chunk: {chunk.page_content[:80]}...")
    else:
        clean_final.append(chunk)

print(f"\nNoise chunks removed : {removed}")
print(f"Final clean chunks   : {len(clean_final)}")
chunks = clean_final

🗑️ Removed noise chunk: We create customized solutions for different entities
We offer quality services ...
🗑️ Removed noise chunk: Elevate your online business with our custom eCommerce solutions. We specialize ...
🗑️ Removed noise chunk: Thank you for your message. It has been sent.
There was an error trying to send ...
🗑️ Removed noise chunk: Transform the beauty experience with AI Virtual Makeup. Let your customers try o...

Noise chunks removed : 4
Final clean chunks   : 21


In [9]:
# Free local model — no API key needed
# Downloads ~90MB on first run, then cached

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("✅ Embedding model loaded")

# Test it
test_vector = embeddings.embed_query("What services does CrocoIT offer?")
print(f"Vector size: {len(test_vector)}")

c:\Users\ahmed\anaconda3\envs\ml_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Embedding model loaded
Vector size: 384


In [10]:
print("Building FAISS index... (may take 1-2 minutes)")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"✅ FAISS index built with {len(chunks)} vectors")

Building FAISS index... (may take 1-2 minutes)
✅ FAISS index built with 21 vectors


In [11]:
os.makedirs("vector_store", exist_ok=True)

vector_store.save_local("vector_store")

print("✅ Vector store saved to vector_store/")

✅ Vector store saved to vector_store/


In [ ]:
# Test that the retriever actually works
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

test_query = "What solutions does CrocoIT offer?"
results = retriever.get_relevant_documents(test_query)

print(f"Query: {test_query}")
print(f"Found {len(results)} relevant chunks:\n")
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['url']}")
    print(doc.page_content[:300])
    print()




Query: What solutions does CrocoIT offer?
Found 5 relevant chunks:

--- Chunk 1 ---
Source: https://crocoit.com/about-us/
Efficiency and increasing ROI are our ways to support different businesses; no more paying for extra features you don’t even want. With CrocoIT, you will only pay for the features you require and get support on much more for free, without extra charges!

--- Chunk 2 ---
Source: https://crocoit.com/about-us/
CrocoIT is one of the fastest growing software houses
Start your business with CrocoIT
We are ready to design and develop a custom e-commerce platform (online Store ), web application, commerce mobiles apps and mobile applications for Android and ios. We are ready to build amazing web and mobile app

--- Chunk 3 ---
Source: https://crocoit.com
Some kind words from our respected clients
Working with CrocoIT was (and continues to be) a great experience. Our Website conversions have just improved to 35% since relaunching our website with CrocoIT experts’ recommendat